In [ ]:
# ============================================================
# BUILD DIM_DRIVERS — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql import functions as F
import nbformat
from nbconvert import PythonExporter

# p_batch_id = "20250101_090000"

def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/04_gold_helpers.ipynb")

In [ ]:
# -----------------------------------------
# p_batch_id
# -----------------------------------------
if "p_batch_id" in globals() and p_batch_id not in [None, ""]:
    pass
elif "dbutils" in globals():
    p_batch_id = dbutils.widgets.get("p_batch_id")
else:
    raise Exception("❌ p_batch_id not provided")

print("Using p_batch_id:", p_batch_id)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 23:09:29 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/08 23:09:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 23:09:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 23:09:30 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/08 23:09:30 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/08 23:09:30 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/08 23:09:30 

✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [ ]:
# -----------------------------------------
# Paths
# -----------------------------------------
silver_drivers = f"{SILVER_PATH}/drivers/data.parquet"
gold_ref = f"{GOLD_PATH}/ref_nationality_region/data.parquet"

gold_path = f"{GOLD_PATH}/dim_drivers"
os.makedirs(gold_path, exist_ok=True)

# Read
drivers_df = spark.read.parquet(silver_drivers).filter(F.col("batch_id") == p_batch_id)
ref_df = spark.read.parquet(gold_ref)

if drivers_df.count() == 0:
    raise Exception("❌ Silver drivers has 0 rows for this batch_id")

In [ ]:
# -----------------------------------------
# Join
# -----------------------------------------
dim_drivers_df = (
    drivers_df.alias("d")
        .join(
            ref_df.alias("r"),
            F.col("d.nationality") == F.col("r.nationality"),
            "left"
        )
        .select(
            F.col("d.driver_id"),
            F.col("d.driver_name"),
            F.col("d.date_of_birth"),
            F.col("d.nationality"),
            F.col("r.region").alias("nationality_region")
        )
)

In [1]:
# -----------------------------------------
# Write
# -----------------------------------------
write_to_gold(
    input_df=dim_drivers_df,
    target_path=f"{gold_path}/data.parquet",
    merge_keys=["driver_id"]
)

print("✔ dim_drivers written")

spark.read.parquet(f"{gold_path}/data.parquet").show(5, truncate=False)

NameError: name 'write_to_gold' is not defined